In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from src.models.lstm_model import LSTMPredictor, save_model
from src.models.xgboost_model import XGBoostPredictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Part 1: LSTM Training

In [ ]:
class CMAPSSDataset(Dataset):
    """Sliding window dataset for LSTM training."""
    def __init__(self, df, feature_cols, seq_len=30):
        self.sequences = []
        self.targets = []
        for _, group in df.groupby('engine_id'):
            X = group[feature_cols].values.astype(np.float32)
            y = group['RUL'].values.astype(np.float32)
            for i in range(len(X)):
                if i < seq_len:
                    pad = np.zeros((seq_len - i - 1, X.shape[1]), dtype=np.float32)
                    seq = np.vstack([pad, X[:i+1]])
                else:
                    seq = X[i-seq_len+1:i+1]
                self.sequences.append(seq)
                self.targets.append(y[i])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.tensor(self.targets[idx])

In [ ]:
train_df = pd.read_parquet('../data/processed/train.parquet')
val_df   = pd.read_parquet('../data/processed/val.parquet')

feature_cols = [c for c in train_df.columns if c.endswith('_mean') or c.endswith('_std')]
print(f"Features: {len(feature_cols)}")

train_ds = CMAPSSDataset(train_df, feature_cols)
val_ds   = CMAPSSDataset(val_df,   feature_cols)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0)
print(f"Train samples: {len(train_ds):,} | Val samples: {len(val_ds):,}")

In [ ]:
model     = LSTMPredictor(input_size=len(feature_cols)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5, verbose=True)
criterion = nn.MSELoss()

best_val_loss = float('inf')
history = {'train': [], 'val': []}
EPOCHS  = 100

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_train = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_train += loss.item()
    train_loss = total_train / len(train_loader)

    # --- Validate ---
    model.eval()
    with torch.no_grad():
        val_loss = sum(
            criterion(model(xb.to(device)), yb.to(device)).item()
            for xb, yb in val_loader
        ) / len(val_loader)

    scheduler.step(val_loss)
    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_model(model, '../models/lstm_model.pt')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

print(f"\nBest Val Loss: {best_val_loss:.4f}")
print("Saved → models/lstm_model.pt")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history['train'], label='Train MSE')
plt.plot(history['val'],   label='Val MSE')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/lstm_training_curve.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 2: XGBoost Training
XGBoost operates on tabular features (last-window rolling stats per engine).

In [ ]:
def get_last_window(df, feature_cols):
    """Return last-row feature vector per engine (XGBoost input)."""
    last = df.groupby('engine_id').last().reset_index()
    return last[feature_cols].values, last['RUL'].values

X_train, y_train = get_last_window(train_df, feature_cols)
X_val,   y_val   = get_last_window(val_df,   feature_cols)
print(f"XGBoost train shape: {X_train.shape}")

In [ ]:
xgb_model = XGBoostPredictor()
xgb_model.fit(X_train, y_train)
xgb_model.save('../models/xgboost_model.pkl')

val_pred = xgb_model.predict(X_val)
val_rmse = np.sqrt(np.mean((val_pred - y_val) ** 2))
print(f"XGBoost Val RMSE: {val_rmse:.2f} cycles")
print("Saved → models/xgboost_model.pkl")

In [ ]:
importances = xgb_model.get_feature_importance()
top_idx = np.argsort(importances)[-15:]

plt.figure(figsize=(8, 6))
plt.barh([feature_cols[i] for i in top_idx], importances[top_idx], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('XGBoost — Top 15 Features')
plt.tight_layout()
plt.savefig('../docs/xgb_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary
Both models are trained and saved to `models/`:
- `lstm_model.pt` — best validation checkpoint
- `xgboost_model.pkl` — main model + 20 bootstrap models
- `scaler.pkl` — already saved in notebook 02

Run **04_Evaluation.ipynb** next to compute official test-set metrics.